# 🐘 Yaya-125M Pretraining — Kaggle T4

**What this does:**
1. Clones the repo and installs dependencies
2. Downloads 3 corpora (~350M+ tokens): WikiText-103, TinyStories, C4-en
3. Tokenizes into binary shards
4. Trains for ~7K steps per session (40K total across ~6 sessions)
5. Pushes checkpoints to HuggingFace Hub for resume

**Before running:**
- Add `HF_TOKEN` to Kaggle Secrets (Settings → Add-ons → Secrets)
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Set Internet: ON

**Just run all cells — it handles everything automatically.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Install dependencies and clone repo
# ═══════════════════════════════════════════════════════════════
!pip install -q sentencepiece datasets huggingface_hub pyyaml torch numpy

import os
REPO_DIR = '/kaggle/working/yaya-ai'

if os.path.isdir(REPO_DIR):
    print('Repo exists — pulling latest...')
    !cd {REPO_DIR} && git pull --ff-only
else:
    print('Cloning repo...')
    !git clone https://github.com/jaylink-coder/miss-yaya.git /kaggle/working/temp-clone
    # The repo has yaya-ai as a subdirectory
    !mv /kaggle/working/temp-clone/yaya-ai {REPO_DIR}
    !rm -rf /kaggle/working/temp-clone

print(f'Repo ready at {REPO_DIR}')
!ls {REPO_DIR}

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Set up HuggingFace token
# ═══════════════════════════════════════════════════════════════
from kaggle_secrets import UserSecretsClient
try:
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    print('✓ HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'⚠ Could not load HF_TOKEN: {e}')
    print('Checkpoints will NOT persist across sessions!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Verify GPU and environment
# ═══════════════════════════════════════════════════════════════
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'bf16 support: {torch.cuda.is_bf16_supported()}')

# Verify key files exist
REPO_DIR = '/kaggle/working/yaya-ai'
for f in ['scripts/kaggle_pretrain.py', 'scripts/train.py', 
          'data/tokenizer/yaya_tokenizer.model', 'configs/model/yaya_125m.yaml']:
    path = os.path.join(REPO_DIR, f)
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'{status} {f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: RUN PRETRAINING (this is the main cell)
# ═══════════════════════════════════════════════════════════════
# This cell:
#   - Downloads WikiText-103 + TinyStories + C4-en (~350M tokens)
#   - Resumes from last Hub checkpoint (if any)
#   - Trains for ~7000 steps (~3 hours)
#   - Pushes checkpoint to HuggingFace Hub
#
# Run this cell each Kaggle session. It auto-resumes.
# ~6 sessions to complete all 40K steps.

REPO_DIR = '/kaggle/working/yaya-ai'
!cd {REPO_DIR} && python -u scripts/kaggle_pretrain.py

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: (Optional) Check training progress
# ═══════════════════════════════════════════════════════════════
import json, glob, os

CKPT_DIR = '/kaggle/working/yaya-pretrain-checkpoints'
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')))
print(f'Local checkpoints: {len(ckpts)}')
for c in ckpts:
    step = os.path.basename(c).split('-')[-1]
    size_mb = sum(os.path.getsize(os.path.join(c, f)) for f in os.listdir(c) if os.path.isfile(os.path.join(c, f))) / 1e6
    print(f'  {os.path.basename(c)}: {size_mb:.0f} MB')

# Check Hub progress
try:
    from huggingface_hub import hf_hub_download
    p = hf_hub_download('Jaylink-coder/yaya-125m', 'pretrain_progress.json', 
                        repo_type='model', token=os.environ.get('HF_TOKEN'))
    progress = json.load(open(p))
    step = progress.get('step', 0)
    total = progress.get('total_steps', 40000)
    pct = step / total * 100
    print(f'\nHub progress: step {step:,}/{total:,} ({pct:.0f}%)')
    remaining_sessions = max(0, (total - step) / 7000)
    print(f'Estimated sessions remaining: {remaining_sessions:.1f}')
except:
    print('No Hub progress found — first session?')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: (Optional) Quick generation test
# ═══════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/kaggle/working/yaya-ai')

import torch
from src.model.transformer import YayaTransformer
from src.tokenizer.tokenizer import YayaTokenizer
from src.utils.config import ModelConfig
import yaml, glob, os

# Load latest checkpoint
CKPT_DIR = '/kaggle/working/yaya-pretrain-checkpoints'
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'checkpoint-*')))
if ckpts:
    ckpt = ckpts[-1]
    print(f'Loading {os.path.basename(ckpt)}...')
    
    cfg = yaml.safe_load(open('/kaggle/working/yaya-ai/configs/model/yaya_125m.yaml'))
    model_cfg = ModelConfig(**cfg)
    model = YayaTransformer(model_cfg)
    state = torch.load(os.path.join(ckpt, 'model.pt'), map_location='cpu')
    model.load_state_dict(state)
    model.eval().cuda()
    
    tokenizer = YayaTokenizer('/kaggle/working/yaya-ai/data/tokenizer/yaya_tokenizer.model')
    
    prompts = ['The capital of France is', 'Once upon a time', 'Water boils at']
    for prompt in prompts:
        ids = tokenizer.encode(prompt)
        x = torch.tensor([ids], device='cuda')
        with torch.no_grad():
            for _ in range(50):
                logits = model(x)[:, -1, :]
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
                x = torch.cat([x, next_id], dim=1)
        text = tokenizer.decode(x[0].tolist())
        print(f'\n> {prompt}')
        print(f'  {text}')
else:
    print('No checkpoints found — run Cell 4 first')